In [1]:
# ============================================================
# CELL 1: SETUP + IMPORT + CONFIG (STSB)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets==2.21.0 scikit-learn scipy psutil

import os
import re
import csv
import json
import time
import random
import logging

from pathlib import Path

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence
)

from torch.optim import AdamW

from datasets import load_dataset

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)

from scipy.stats import pearsonr, spearmanr

# ============================================================
# OUTPUT
# ============================================================

OUTPUT_ROOT = "/content/drive/MyDrive/STL_BILSTM_STSB_RESULTS"

Path(OUTPUT_ROOT).mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("DEVICE:", DEVICE)

# ============================================================
# HP
# ============================================================

HP = {

    "batch_size": 32,

    "eval_batch_size": 64,

    "learning_rate": 1e-3,

    "weight_decay": 1e-4,

    "max_epochs": 10,

    "patience": 3,

    "max_seq_length": 128,

    "embed_dim": 256,

    "hidden": 256,

    "num_layers": 2,

    "dropout": 0.3,

    "vocab_size": 50000,

    "num_workers": 2,

    "seed": 42,

    "grad_clip": 1.0,

    "keep_n_ckpts": 2,

    "bench_max_batches": 50,
}

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

# ============================================================
# LOGGER
# ============================================================

def make_logger(name, log_path):

    logger = logging.getLogger(name)

    logger.setLevel(logging.INFO)

    logger.propagate = False

    for h in list(logger.handlers):

        logger.removeHandler(h)

    fmt = logging.Formatter(
        "[%(asctime)s] %(message)s"
    )

    fh = logging.FileHandler(log_path)

    fh.setFormatter(fmt)

    sh = logging.StreamHandler()

    sh.setFormatter(fmt)

    logger.addHandler(fh)

    logger.addHandler(sh)

    return logger

# ============================================================
# HISTORY
# ============================================================

class HistoryWriter:

    def __init__(self, out_dir):

        self.csv_path = Path(out_dir) / "history.csv"

        self.records = []

        self.fields = []

    def append(self, row):

        for k in row:

            if k not in self.fields:

                self.fields.append(k)

        self.records.append(row)

        with open(
            self.csv_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=self.fields
            )

            writer.writeheader()

            for r in self.records:

                writer.writerow(r)

# ============================================================
# TOKENIZER
# ============================================================

def tokenize(text):

    text = str(text).lower()

    text = re.sub(r"[^a-z0-9 ]+", " ", text)

    return text.split()

# ============================================================
# VOCAB
# ============================================================

class Vocab:

    def __init__(self):

        self.stoi = {
            "<pad>": 0,
            "<unk>": 1
        }

        self.itos = [
            "<pad>",
            "<unk>"
        ]

    def build(self, corpus, max_size):

        freq = {}

        for text in corpus:

            for tok in tokenize(text):

                freq[tok] = freq.get(tok, 0) + 1

        items = sorted(
            freq.items(),
            key=lambda x: x[1],
            reverse=True
        )

        items = items[:max_size]

        for w, _ in items:

            if w not in self.stoi:

                self.stoi[w] = len(self.itos)

                self.itos.append(w)

    def encode(self, text):

        return [

            self.stoi.get(tok, 1)

            for tok in tokenize(text)
        ]

    def __len__(self):

        return len(self.itos)

# ============================================================
# DATASET
# ============================================================

class STSBDataset(Dataset):

    def __init__(
        self,
        split,
        vocab,
        max_len
    ):

        self.s1 = split["sentence1"]

        self.s2 = split["sentence2"]

        self.labels = split["label"]

        self.vocab = vocab

        self.max_len = max_len

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        text = (
            str(self.s1[idx]) +
            " [SEP] " +
            str(self.s2[idx])
        )

        ids = self.vocab.encode(
            text
        )[:self.max_len]

        return {

            "ids":
                torch.tensor(ids),

            "label":
                torch.tensor(
                    float(self.labels[idx]),
                    dtype=torch.float
                )
        }

# ============================================================
# COLLATE
# ============================================================

def collate(batch):

    ids = [
        x["ids"]
        for x in batch
    ]

    labels = torch.stack([
        x["label"]
        for x in batch
    ])

    lens = torch.tensor([
        len(x)
        for x in ids
    ])

    ids = pad_sequence(
        ids,
        batch_first=True,
        padding_value=0
    )

    return {

        "ids": ids,

        "lens": lens,

        "labels": labels,
    }

print("CELL 1 DONE")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
DEVICE: cuda
CELL 1 DONE


In [2]:
# ============================================================
# CELL 2: STL_BILSTM STSB
# ============================================================

class BiLSTM(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.emb = nn.Embedding(
            vocab_size,
            HP["embed_dim"],
            padding_idx=0
        )

        self.lstm = nn.LSTM(

            HP["embed_dim"],

            HP["hidden"],

            num_layers=HP["num_layers"],

            batch_first=True,

            bidirectional=True,

            dropout=HP["dropout"]
        )

        self.drop = nn.Dropout(
            HP["dropout"]
        )

        self.fc = nn.Linear(
            HP["hidden"] * 2,
            1
        )

    def forward(
        self,
        ids,
        lens
    ):

        x = self.emb(ids)

        packed = pack_padded_sequence(

            x,

            lens.cpu(),

            batch_first=True,

            enforce_sorted=False
        )

        _, (h, _) = self.lstm(packed)

        h = torch.cat(
            [h[-2], h[-1]],
            dim=1
        )

        h = self.drop(h)

        return self.fc(h).squeeze(1)

# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    preds = []

    labels = []

    losses = []

    mse_loss = nn.MSELoss()

    for batch in loader:

        ids = batch["ids"].to(DEVICE)

        lens = batch["lens"].to(DEVICE)

        y = batch["labels"].to(DEVICE)

        outputs = model(ids, lens)

        loss = mse_loss(outputs, y)

        losses.append(loss.item())

        preds.extend(
            outputs.cpu().numpy()
        )

        labels.extend(
            y.cpu().numpy()
        )

    preds = np.array(preds)

    labels = np.array(labels)

    pearson = pearsonr(labels, preds)[0]

    spearman = spearmanr(labels, preds)[0]

    mse = mean_squared_error(labels, preds)

    mae = mean_absolute_error(labels, preds)

    return {

        "pearson":
            pearson,

        "spearman":
            spearman,

        "mse":
            mse,

        "mae":
            mae,

        "eval_loss":
            np.mean(losses),
    }

# ============================================================
# BENCHMARK
# ============================================================

@torch.no_grad()
def benchmark(model, loader):

    model.eval()

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats()

        torch.cuda.synchronize()

    t0 = time.perf_counter()

    n = 0

    for i, batch in enumerate(loader):

        if i >= HP["bench_max_batches"]:
            break

        ids = batch["ids"].to(DEVICE)

        lens = batch["lens"].to(DEVICE)

        _ = model(ids, lens)

        n += ids.shape[0]

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    vram = (

        torch.cuda.max_memory_allocated()
        / 1024**2

        if torch.cuda.is_available()

        else 0.0
    )

    return {

        "inference_latency_ms":
            (elapsed / max(n, 1)) * 1000,

        "throughput_samples_per_sec":
            n / max(elapsed, 1e-9),

        "peak_vram_mb":
            vram,
    }

# ============================================================
# GET LATEST CHECKPOINT
# ============================================================

def get_latest_checkpoint(ckpt_dir):

    ckpts = sorted(
        ckpt_dir.glob(
            "checkpoint_epoch_*.pt"
        )
    )

    if len(ckpts) == 0:

        return None

    return ckpts[-1]

# ============================================================
# RUN
# ============================================================

def run():

    set_seed(HP["seed"])

    out_dir = Path(OUTPUT_ROOT)

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    ckpt_dir = out_dir / "checkpoints"

    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = make_logger(
        "train",
        str(out_dir / "train.log")
    )

    # ========================================================
    # LOAD DATASET
    # ========================================================

    raw = load_dataset(
        "glue",
        "stsb"
    )

    # ========================================================
    # BUILD VOCAB
    # ========================================================

    corpus = []

    for s1, s2 in zip(
        raw["train"]["sentence1"],
        raw["train"]["sentence2"]
    ):

        corpus.append(
            str(s1) + " " + str(s2)
        )

    vocab = Vocab()

    vocab.build(
        corpus,
        HP["vocab_size"]
    )

    # ========================================================
    # DATASET
    # ========================================================

    train_ds = STSBDataset(
        raw["train"],
        vocab,
        HP["max_seq_length"]
    )

    val_ds = STSBDataset(
        raw["validation"],
        vocab,
        HP["max_seq_length"]
    )

    # ========================================================
    # DATALOADER
    # ========================================================

    train_dl = DataLoader(

        train_ds,

        batch_size=HP["batch_size"],

        shuffle=True,

        collate_fn=collate,

        num_workers=HP["num_workers"]
    )

    val_dl = DataLoader(

        val_ds,

        batch_size=HP["eval_batch_size"],

        shuffle=False,

        collate_fn=collate,

        num_workers=HP["num_workers"]
    )

    # ========================================================
    # MODEL
    # ========================================================

    model = BiLSTM(
        len(vocab)
    ).to(DEVICE)

    opt = AdamW(

        model.parameters(),

        lr=HP["learning_rate"],

        weight_decay=HP["weight_decay"]
    )

    criterion = nn.MSELoss()

    start_epoch = 1

    best = -999

    # ========================================================
    # RESUME CHECKPOINT
    # ========================================================

    latest_ckpt = get_latest_checkpoint(
        ckpt_dir
    )

    if latest_ckpt is not None:

        print(
            "RESUME:",
            latest_ckpt
        )

        ckpt = torch.load(
            latest_ckpt,
            map_location=DEVICE
        )

        model.load_state_dict(
            ckpt["model"]
        )

        opt.load_state_dict(
            ckpt["optimizer"]
        )

        start_epoch = (
            ckpt["epoch"] + 1
        )

        best = ckpt["best"]

    # ========================================================
    # HISTORY
    # ========================================================

    hist = HistoryWriter(out_dir)

    total_start = time.perf_counter()

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for epoch in range(

        start_epoch,

        HP["max_epochs"] + 1
    ):

        model.train()

        losses = []

        t0 = time.perf_counter()

        for batch in train_dl:

            ids = batch["ids"].to(DEVICE)

            lens = batch["lens"].to(DEVICE)

            y = batch["labels"].to(DEVICE)

            opt.zero_grad()

            outputs = model(ids, lens)

            loss = criterion(outputs, y)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                HP["grad_clip"]
            )

            opt.step()

            losses.append(
                loss.item()
            )

        epoch_time = (
            time.perf_counter()
            - t0
        )

        metrics = evaluate(
            model,
            val_dl
        )

        bench = benchmark(
            model,
            val_dl
        )

        row = {

            "epoch":
                epoch,

            "train_loss":
                np.mean(losses),

            "eval_loss":
                metrics["eval_loss"],

            "pearson":
                metrics["pearson"],

            "spearman":
                metrics["spearman"],

            "mse":
                metrics["mse"],

            "mae":
                metrics["mae"],

            "inference_latency_ms":
                bench["inference_latency_ms"],

            "throughput_samples_per_sec":
                bench["throughput_samples_per_sec"],

            "peak_vram_mb":
                bench["peak_vram_mb"],

            "time_per_epoch":
                epoch_time,
        }

        hist.append(row)

        logger.info(row)

        # ====================================================
        # SAVE CHECKPOINT
        # ====================================================

        torch.save(

            {

                "epoch":
                    epoch,

                "model":
                    model.state_dict(),

                "optimizer":
                    opt.state_dict(),

                "best":
                    best,
            },

            ckpt_dir / f"checkpoint_epoch_{epoch}.pt"
        )

        # ====================================================
        # BEST MODEL
        # ====================================================

        if metrics["pearson"] > best:

            best = metrics["pearson"]

            torch.save(

                model.state_dict(),

                out_dir / "best_model.pt"
            )

            print(
                "NEW BEST:",
                best
            )

    print("\nDONE")

run()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

[2026-05-19 05:13:36,691] {'epoch': 1, 'train_loss': np.float64(2.3138955351379185), 'eval_loss': np.float64(2.2190516889095306), 'pearson': np.float32(0.16820697), 'spearman': np.float64(0.15051022219611288), 'mse': 2.2053980827331543, 'mae': 1.254658579826355, 'inference_latency_ms': 0.1730198239999936, 'throughput_samples_per_sec': 5779.684529097874, 'peak_vram_mb': 189.380859375, 'time_per_epoch': 4.507134390000033}


...

NEW BEST: 0.19987714


[2026-05-19 05:13:50,552] {'epoch': 4, 'train_loss': np.float64(0.8917096419466867), 'eval_loss': np.float64(2.5266358703374863), 'pearson': np.float32(0.193085), 'spearman': np.float64(0.1909409501060004), 'mse': 2.5254271030426025, 'mae': 1.2917970418930054, 'inference_latency_ms': 0.18168554466668257, 'throughput_samples_per_sec': 5504.0152029407955, 'peak_vram_mb': 189.3818359375, 'time_per_epoch': 3.7785353310000005}
[2026-05-19 05:13:54,779] {'epoch': 5, 'train_loss': np.float64(0.5715388043059243), 'eval_loss': np.float64(2.6782917430003486), 'pearson': np.float32(0.18650171), 'spearman': np.float64(0.19027123078320526), 'mse': 2.676231861114502, 'mae': 1.3237429857254028, 'inference_latency_ms': 0.18658811000000242, 'throughput_samples_per_sec': 5359.398302496269, 'peak_vram_mb': 189.38134765625, 'time_per_epoch': 3.497061171000041}
[2026-05-19 05:14:00,253] {'epoch': 6, 'train_loss': np.float64(0.39846708981527224), 'eval_loss': np.float64(2.5812813142935433), 'pearson': np.fl

NEW BEST: 0.21957117


[2026-05-19 05:14:04,453] {'epoch': 7, 'train_loss': np.float64(0.2911241294609176), 'eval_loss': np.float64(2.7025142461061478), 'pearson': np.float32(0.20010157), 'spearman': np.float64(0.2015825659813591), 'mse': 2.699666976928711, 'mae': 1.3247231245040894, 'inference_latency_ms': 0.1754220793333161, 'throughput_samples_per_sec': 5700.536692989024, 'peak_vram_mb': 189.38232421875, 'time_per_epoch': 3.4126090849999855}
[2026-05-19 05:14:08,848] {'epoch': 8, 'train_loss': np.float64(0.23303141734666294), 'eval_loss': np.float64(2.6349017545580864), 'pearson': np.float32(0.20967968), 'spearman': np.float64(0.21083418445742025), 'mse': 2.6405951976776123, 'mae': 1.3091826438903809, 'inference_latency_ms': 0.1700818746666831, 'throughput_samples_per_sec': 5879.521271503762, 'peak_vram_mb': 189.38037109375, 'time_per_epoch': 3.6697273500000165}
[2026-05-19 05:14:14,106] {'epoch': 9, 'train_loss': np.float64(0.20121357556846406), 'eval_loss': np.float64(2.5286927173535028), 'pearson': np.


DONE
